# Gvial: estructura, intermediación y corredores críticos

**Pregunta:** ¿qué tramos de la red vial arterial de Bogotá concentran el flujo y qué tan dependientes son de la velocidad?

Este notebook reproduce, con el módulo común `src/pipeline_redes.py`, las cifras de estructura, intermediación, concentración y dependencia de velocidad de la presentación de avance. La última sección compara cada cifra contra la línea base del notebook histórico (`referencia_avance.csv`).

Corre igual en Google Colab (lee de Google Drive) y en un equipo local (lee de `data/`).

## 1. Entorno y rutas

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Versiones fijadas en la decisión A4
VERSIONES = {"osmnx": "2.1.1", "networkx": "3.7"}

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[f"{p}=={v}" for p, v in VERSIONES.items()]], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/proyecto-redes-bogota")
    RUTAS = {
        "src": BASE / "src",
        "grafos": BASE / "data" / "processed",
        "salida": BASE / "resultados" / "gvial",
        "referencia": BASE / "resultados" / "referencia_avance.csv",
    }
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUTAS = {
        "src": RAIZ / "src",
        "grafos": RAIZ / "data" / "drive_snapshot" / "processed",
        "salida": RAIZ / "data" / "salidas_locales" / "resultados" / "gvial",
        "referencia": RAIZ / "data" / "linea_base" / "salida" / "resultados" / "referencia_avance.csv",
    }

RUTAS["salida"].mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RUTAS["src"]))
for nombre, ruta in RUTAS.items():
    print(f"{nombre:10s} {ruta}")

In [ ]:
import networkx as nx
import osmnx as ox
import pandas as pd

import pipeline_redes as pr

for paquete, version in VERSIONES.items():
    instalada = sys.modules[paquete].__version__
    aviso = "" if instalada == version else f"  <- se esperaba {version} (decisión A4)"
    print(f"{paquete} {instalada}{aviso}")

## 2. Parámetros del análisis

Todos los parámetros están aquí. Los valores reproducen el notebook del avance; las decisiones que los cambian se citan al lado.

In [ ]:
PARAM = {
    "semilla": 42,
    "n_fuentes_estructura": 500,   # nodos de origen para el tiempo medio de viaje
    "k_intermediacion": 1000,      # fuentes muestreadas (A7: Gini y 1% superior usan k=1000)
    "min_km_corredor": 2.0,        # corredores más cortos se tratan como fragmentos
    "top_n": 10,
    # Cortes del índice de dependencia de velocidad
    "cortes_avance": [1.5, 3],     # celda 18 del notebook histórico, solo para verificar
    "etiquetas_avance": ["estructural", "mixto", "por velocidad"],
    "cortes_a6": [0.6, 2],         # decisión A6
    "etiquetas_a6": ["subutilizados", "estructurales", "atraen por velocidad"],
    "reusar_red_completa": True,   # usa tramos_completa.csv si ya existe (ahorra unos 15 min)
    # Eficiencia, cierres y ataques (valores del avance)
    "n_fuentes_eficiencia": 300,
    "corredores_cierre": ["Avenida Carrera 30", "Avenida Carrera 72", "Avenida Carrera 68",
                          "Avenida Carrera 86", "Avenida Carrera 45", "Avenida Calle 26",
                          "Transversal 35", "Avenida Calle 45A Sur"],
    "n_control_cierre": 5,
    "ataque": {"pasos": 15, "frac": 0.01, "k_recalculo": 300, "reps_azar": 5},
    "paso_cascada": 7,
    "umbral_fc": 0.5,                                  # A5
    "ataque_informe": {"pasos": 30, "frac": 0.005},    # A5: pasos de 0,5%
    # Intervenciones (definición de escenarios, no resultados)
    "interv": {
        "zona_a": 0,   # macrozona centro-occidente (la mayor tras 7 pasos de ataque)
        "zona_b": 2,   # macrozona sur (la tercera)
        "n_candidatos": 3,
        # A: enlace de 4,2 km elegido en el avance (tercer candidato por eficiencia por km)
        "enlace_A": (261630424, 261040900),
        "vel_kph": 50,
        "factor": 1.2,
        "corredores_B": ["Avenida Carrera 68", "Avenida Carrera 50"],
        "robustez": {"pasos": 10, "frac": 0.01, "k": 300},
        "offsets_replicas": [0, 100, 200],
    },
}
CORREDORES_CITADOS = {
    "NQS": "Avenida Carrera 30",
    "Transversal 35": "Transversal 35",
    "Av. Ciudad de Cali": "Avenida Carrera 86",
    "Av. Boyacá": "Avenida Carrera 72",
    "Carrera 68": "Avenida Carrera 68",
    "Carrera 50": "Avenida Carrera 50",
}

## 3. Adaptador de Gvial

Convierte el `.graphml` de OSMnx al contrato del núcleo: grafo dirigido sin aristas paralelas (se conserva la de menor tiempo), nombres como texto y atributos de origen. Los grafos son los del 2026-10-05 (decisión A3); no se descargan de nuevo porque OSM cambia con el tiempo.

In [ ]:
def cargar_gvial(nombre_archivo: str, red: str) -> tuple[nx.MultiDiGraph, nx.DiGraph]:
    """Carga un .graphml del snapshot y devuelve el grafo original y el grafo del contrato."""
    rutas = sorted(RUTAS["grafos"].glob(f"{nombre_archivo}.graphml*"))
    if len(rutas) != 1:
        raise FileNotFoundError(f"Se esperaba un {nombre_archivo}.graphml en {RUTAS['grafos']}, hay {len(rutas)}")
    G_osm = ox.load_graphml(rutas[0])
    G = ox.convert.to_digraph(G_osm, weight="travel_time")
    for _, _, d in G.edges(data=True):
        d["name"] = pr.normalizar_nombre(d.get("name"))
    G.graph.update(
        red=red,
        crs=str(G_osm.graph.get("crs")),
        fuente=f"OpenStreetMap vía {G_osm.graph.get('created_with')}",
        fecha_snapshot=G_osm.graph.get("created_date"),
    )
    pr.validar_contrato(G)
    return G_osm, G


G_art_osm, G_art = cargar_gvial("gvial_arterial", "Gvial arterial")
G_drv_osm, G_drv = cargar_gvial("gvial_drive", "Gvial completa")

for nombre, G_osm, G in (("arterial", G_art_osm, G_art), ("completa", G_drv_osm, G_drv)):
    print(f"{nombre}: .graphml {G_osm.number_of_nodes()} nodos y {G_osm.number_of_edges()} aristas; "
          f"contrato {G.number_of_nodes()} nodos y {G.number_of_edges()} aristas; "
          f"snapshot {G.graph['fecha_snapshot']}")

## 4. Estructura

In [ ]:
est = pr.estructura(G_art, n_fuentes=PARAM["n_fuentes_estructura"], seed=PARAM["semilla"])
print(f"N={est['N']}, M={est['M']}, grado medio={est['grado_medio']:.2f}")
print(f"Clustering medio={est['clustering']:.4f}")
print(f"Tiempo medio de viaje (flujo libre): {est['tiempo_medio_min']:.1f} min")

## 5. Intermediación y concentración

Se calcula la intermediación de aristas ponderada por tiempo y por distancia, muestreada (k=1000) y exacta. La red arterial admite el cálculo exacto en pocos minutos.

In [ ]:
k, s = PARAM["k_intermediacion"], PARAM["semilla"]
t0 = time.time()
eb = {
    "eb_k": pr.intermediacion(G_art, "travel_time", k=k, seed=s),
    "eb_len_k": pr.intermediacion(G_art, "length", k=k, seed=s),
    "eb": pr.intermediacion(G_art, "travel_time"),
    "eb_len": pr.intermediacion(G_art, "length"),
}
print(f"Intermediación de la red arterial: {(time.time() - t0) / 60:.1f} min")

tramos = pr.tabla_tramos(G_art, eb)
tramos.to_csv(RUTAS["salida"] / "tramos_arterial.csv", index=False)

conc = pr.concentracion(tramos["eb_k"])
print(f"Gini de edge betweenness: {conc['gini']:.3f}")
print(f"El 1 % de tramos concentra el {conc['top1']:.1%} de la intermediación total")

top_k = tramos.groupby("nombre")["eb_k"].max().idxmax()
print("Corredor con el tramo de mayor intermediación (k=1000):", top_k)

## 6. Corredores y dependencia de velocidad

La carga de un corredor es su intermediación media ponderada por longitud. El índice de dependencia de velocidad divide la carga por tiempo entre la carga por distancia.

In [ ]:
corr = pr.corredores(tramos, {"carga_t": "eb", "carga_l": "eb_len"}, min_km=PARAM["min_km_corredor"])
excluidos = pr.tramos_excluidos(tramos)
print(f"Sin nombre (fuera del análisis por corredor): {excluidos['tramos']} tramos, {excluidos['km']:.1f} km")

jac = pr.jaccard_top(corr["carga_t"], corr["carga_l"], n=PARAM["top_n"])
rho_pesos = pr.spearman(tramos["eb"], tramos["eb_len"])
print(f"Jaccard corredores: {jac:.2f}")
print(f"Spearman tramos (todos): {rho_pesos:.2f}")

top10 = list(corr["carga_t"].nlargest(PARAM["top_n"]).index)
corr.sort_values("carga_t", ascending=False).head(PARAM["top_n"])

In [ ]:
dep_avance = pr.dependencia_velocidad(corr, "carga_t", "carga_l",
                                      PARAM["cortes_avance"], PARAM["etiquetas_avance"])
dep = pr.dependencia_velocidad(corr, "carga_t", "carga_l",
                               PARAM["cortes_a6"], PARAM["etiquetas_a6"])
dep = dep.assign(tipo_avance=dep_avance["tipo"])
dep.to_csv(RUTAS["salida"] / "corredores_arterial.csv")
dep.loc[list(CORREDORES_CITADOS.values()), ["km", "carga_t", "carga_l", "dep_vel", "tipo", "tipo_avance"]]

## 7. Red completa: ¿el filtro arterial sesga el resultado?

Se repite la carga por corredor en la red completa (59.198 intersecciones) con intermediación muestreada y se compara con la red arterial. Este cálculo es el más lento del notebook.

In [ ]:
ruta_drv = RUTAS["salida"] / "tramos_completa.csv"
if PARAM["reusar_red_completa"] and ruta_drv.exists():
    tramos_drv = pd.read_csv(ruta_drv, dtype={"nombre": str}, keep_default_na=False)
    min_drv = float("nan")
    print(f"Se reusa la intermediación ya calculada: {ruta_drv.name}")
else:
    t0 = time.time()
    eb_drv = pr.intermediacion(G_drv, "travel_time", k=PARAM["k_intermediacion"], seed=PARAM["semilla"])
    min_drv = (time.time() - t0) / 60
    print(f"Intermediación de la red completa: {min_drv:.1f} min")
    tramos_drv = pr.tabla_tramos(G_drv, {"eb": eb_drv})
    tramos_drv.to_csv(ruta_drv, index=False)

corr_drv = pr.corredores(tramos_drv, {"carga": "eb"}, min_km=PARAM["min_km_corredor"])

comp = pr.comparar_corredores(corr["carga_t"], corr_drv["carga"], n=PARAM["top_n"])
print(f"Corredores comunes: {comp['comunes']} | Spearman carga: {comp['spearman']:.2f}")
print("Top 10 en ambos:", comp["top_en_ambos"])
print("Solo en la red completa:", comp["solo_en_b"])
corr_drv.sort_values("carga", ascending=False).head(PARAM["top_n"])

## 8. Eficiencia y cierre de corredores

La eficiencia global se estima desde 300 nodos de origen fijos. Cada corredor se cierra completo (ambos sentidos) y se compara con cinco cierres aleatorios de la misma longitud.

In [ ]:
fuentes_ef = pr.muestrear_nodos(G_art, PARAM["n_fuentes_eficiencia"], PARAM["semilla"])
E0 = pr.eficiencia(G_art, fuentes_ef)
print(f"E0 = {E0:.6f}")

cierres = pr.cierre_corredores(G_art, tramos, PARAM["corredores_cierre"], fuentes_ef, E0,
                               n_control=PARAM["n_control_cierre"])
cierres.to_csv(RUTAS["salida"] / "cierre_corredores.csv", index=False)
cierres

## 9. Resiliencia ante ataques

Tres estrategias retiran lotes de 1% de los tramos: al azar (5 réplicas), por ranking fijo de intermediación y por ranking recalculado en cada paso (ataque adaptativo). `f` es la fracción nominal retirada y `f_efectiva` la realmente retirada al cerrar ambos sentidos (A9). La fracción crítica f_c es donde la componente gigante S cae a 0,5 (A5).

In [ ]:
A = PARAM["ataque"]
curvas = []
for s in range(A["reps_azar"]):
    curvas.append(pr.ataque(G_art, "aleatorio", fuentes_ef, E0, A["pasos"], A["frac"], seed=s)[0])
curvas.append(pr.ataque(G_art, "eb_inicial", fuentes_ef, E0, A["pasos"], A["frac"], eb_inicial=eb["eb"])[0])
curva_ad, registro_ad = pr.ataque(G_art, "eb_recalculada", fuentes_ef, E0, A["pasos"], A["frac"],
                                  k_recalculo=A["k_recalculo"])
curvas = pd.concat([*curvas, curva_ad], ignore_index=True)
curvas.to_csv(RUTAS["salida"] / "curvas_resiliencia.csv", index=False)
registro_ad.to_csv(RUTAS["salida"] / "log_ataque_adaptativo.csv", index=False)

azar = curvas[curvas["estrategia"] == "aleatorio"].groupby("f")[["S", "f_efectiva"]].mean()
u = PARAM["umbral_fc"]
fc = {
    "adaptativo": pr.f_critica(curva_ad["f"], curva_ad["S"], u),
    "azar": pr.f_critica(azar.index, azar["S"], u),
}
fc_ef = {
    "adaptativo": pr.f_critica(curva_ad["f_efectiva"], curva_ad["S"], u),
    "azar": pr.f_critica(azar["f_efectiva"], azar["S"], u),
}
caida_5 = 100 * (curva_ad.loc[curva_ad["f"].round(6) == 0.05, "E_rel"].iloc[0] - 1)

for estrategia in fc:
    print(f"f_c {estrategia}: nominal {fc[estrategia]:.1%}, efectiva {fc_ef[estrategia]:.1%}")
print(f"Eficiencia con 5% retirado (adaptativo): {caida_5:+.1f}%")
curvas.pivot_table(index="f", columns="estrategia", values="S").round(3)

## 10. Cascada del ataque adaptativo

Qué corredores retira el ataque adaptativo en cada paso y cómo queda partida la red después del paso 7.

In [ ]:
import json

paso = PARAM["paso_cascada"]
fila = curva_ad[curva_ad["f"].round(6) == round(paso * A["frac"], 6)].iloc[0]
macrozonas = json.loads(fila["componentes_fuertes"])[:3]
debil = fila["debil_mayor"]
print(f"Paso {paso}: macrozonas {macrozonas} nodos; componente débil mayor {debil:.1%}")

retirados = registro_ad[registro_ad["paso"] <= paso].merge(tramos[["u", "v", "nombre"]], on=["u", "v"], how="left")
for p, g in retirados.groupby("paso"):
    print(p, g["nombre"].value_counts().head(4).to_dict())

## 11. Intervenciones A y B

- **A, enlace nuevo:** se buscan pares de intersecciones entre la macrozona centro-occidente y la sur donde un enlace nuevo ahorra más rodeo. El escenario A es el enlace de 4,2 km elegido en el avance (Fontibón a Kennedy); sus nodos están en los parámetros y se verifica que aparezcan entre los candidatos calculados.
- **B, velocidad:** la Carrera 68 y la Carrera 50 pasan a 50 km/h.

Todas las cifras, incluidas las réplicas del índice R, se calculan en esta corrida. Esta sección es la más lenta después de la red completa: cada cálculo de R toma unos 2,5 minutos.

In [ ]:
I = PARAM["interv"]
zonas = pr.componentes_tras_retiro(G_art, registro_ad, PARAM["paso_cascada"])
za, zb = zonas[I["zona_a"]], zonas[I["zona_b"]]
print(f"Zona A: {len(za)} nodos | Zona B: {len(zb)} nodos")

cand = pr.candidatos_costura(G_art, za, zb)
filas = []
for r in cand.itertuples():
    H, km = pr.agregar_enlace(G_art, r.u, r.v, I["vel_kph"], I["factor"])
    dE = 100 * (pr.eficiencia(H, fuentes_ef) / E0 - 1)
    filas.append({"u": r.u, "v": r.v, "km": km, "rodeo": r.rodeo, "dE_%": dE, "dE_por_km": dE / km})
res_cand = pd.DataFrame(filas).sort_values("dE_por_km", ascending=False)
res_cand.to_csv(RUTAS["salida"] / "candidatos_enlace.csv", index=False)

if tuple(I["enlace_A"]) not in set(zip(res_cand["u"], res_cand["v"])):
    raise ValueError("El enlace del escenario A no está entre los candidatos calculados")
res_cand

In [ ]:
rob = I["robustez"]
nombres_B = set(I["corredores_B"])
vel_actual = {n: pd.Series([d["speed_kph"] for _, _, d in G_art.edges(data=True) if d["name"] == n]).mean()
              for n in I["corredores_B"]}
print("Velocidad media actual (km/h):", {n: round(v, 1) for n, v in vel_actual.items()})

escenarios = {}
for i, r in enumerate(res_cand.head(I["n_candidatos"]).itertuples(), 1):
    H, km = pr.agregar_enlace(G_art, r.u, r.v, I["vel_kph"], I["factor"])
    escenarios[f"A{i} enlace {km:.1f} km"] = H
    if (r.u, r.v) == tuple(I["enlace_A"]):
        H_A, km_A, nombre_A = H, km, f"A{i} enlace {km:.1f} km"
H_B = pr.acelerar(G_art, nombres_B, I["vel_kph"])
escenarios["B Cra 68 y Cra 50 a 50 km/h"] = H_B

R0, _ = pr.robustez(G_art, **rob)
print(f"Robustez base R0 = {R0:.3f}")
R_rep0 = {}
filas = []
for nom, H in escenarios.items():
    R_rep0[nom], _ = pr.robustez(H, **rob)
    filas.append({"escenario": nom, "R": R_rep0[nom], "dR_%": 100 * (R_rep0[nom] / R0 - 1),
                  "dE_%": 100 * (pr.eficiencia(H, fuentes_ef) / E0 - 1)})
    print(f"{nom}: R={R_rep0[nom]:.3f}")
res_esc = pd.DataFrame(filas)
res_esc.to_csv(RUTAS["salida"] / "escenarios_robustez.csv", index=False)
print(f"Escenario A: {nombre_A}")
res_esc

In [ ]:
t_base = pr.tiempo_entre_zonas(G_art, za, zb)
tiempo = {nom: 100 * (pr.tiempo_entre_zonas(H, za, zb) / t_base - 1) for nom, H in escenarios.items()}
print(f"Tiempo base entre zonas: {t_base:.1f} min")
for nom, cambio in tiempo.items():
    print(f"{nom}: {cambio:+.1f} %")

k, s, NQS = PARAM["k_intermediacion"], PARAM["semilla"], "Avenida Carrera 30"
carga = lambda G, eb_g: pr.corredores(pr.tabla_tramos(G, {"eb": eb_g}), {"carga": "eb"},
                                      min_km=PARAM["min_km_corredor"])["carga"]
c0 = carga(G_art, eb["eb_k"])
cA = carga(H_A, pr.intermediacion(H_A, "travel_time", k=k, seed=s))
cB = carga(H_B, pr.intermediacion(H_B, "travel_time", k=k, seed=s))
km_B = pr.km_de_corredores(G_art, nombres_B)
carga_nqs = {"A": 100 * (cA[NQS] / c0[NQS] - 1), "B": 100 * (cB[NQS] / c0[NQS] - 1)}
por_km = {"A": carga_nqs["A"] / km_A, "B": carga_nqs["B"] / km_B}
print(f"Carga NQS: A {carga_nqs['A']:+.1f} % ({por_km['A']:+.2f} %/km de {km_A:.1f} km nuevos), "
      f"B {carga_nqs['B']:+.1f} % ({por_km['B']:+.3f} %/km de {km_B:.1f} km intervenidos)")

citados = ["Avenida Carrera 30", "Avenida Carrera 68", "Avenida Carrera 50", "Avenida Carrera 72", "Transversal 35"]
pd.DataFrame({"base": c0, "B": cB}).assign(**{"cambio_%": lambda t: 100 * (t["B"] / t["base"] - 1)}).loc[citados]

In [ ]:
H_AB, _ = pr.agregar_enlace(H_B, *I["enlace_A"], I["vel_kph"], I["factor"])
casos = {"base": G_art, "A": H_A, "B": H_B, "A+B": H_AB}
primera = {"base": R0, "A": R_rep0[nombre_A], "B": R_rep0["B Cra 68 y Cra 50 a 50 km/h"]}   # offset 0, de esta corrida

reps = {}
for nom, H in casos.items():
    reps[nom] = [primera[nom] if (o == 0 and nom in primera) else pr.robustez(H, offset=o, **rob)[0]
                 for o in I["offsets_replicas"]]
    print(nom, [round(r, 4) for r in reps[nom]])

base_reps = pd.Series(reps["base"])
res_final = pd.DataFrame([
    {"escenario": nom, "R_media": pd.Series(r).mean(), "R_std": pd.Series(r).std(ddof=0),
     "dE_%": 100 * (pr.eficiencia(casos[nom], fuentes_ef) / E0 - 1),
     "dR_pareado_%": (100 * (pd.Series(r) - base_reps) / base_reps).mean()}
    for nom, r in reps.items()
])
res_final["dR_%"] = 100 * (res_final["R_media"] / res_final.loc[0, "R_media"] - 1)
res_final.to_csv(RUTAS["salida"] / "escenarios_final.csv", index=False)
res_final

## 12. Verificación contra la línea base

Cada cifra calculada aquí se compara con la del notebook histórico corrido sobre los mismos grafos. Un valor numérico coincide si es igual al redondearlo a los decimales registrados en la línea base. El resultado se guarda en la columna `valor_refactor` de `referencia_avance.csv`.

In [ ]:
dv = dep["dep_vel"]
calculado = {
    "Nodos red arterial": est["N"],
    "Aristas red arterial (DiGraph sin paralelas)": est["M"],
    "Nodos red completa": G_drv_osm.number_of_nodes(),
    "Aristas red completa (.graphml)": G_drv_osm.number_of_edges(),
    "Grado medio": est["grado_medio"],
    "Clustering medio": est["clustering"],
    "Tiempo medio de viaje flujo libre (min)": est["tiempo_medio_min"],
    "Gini de intermediación de aristas": conc["gini"],
    "Porcentaje de intermediación del 1% superior de tramos": 100 * conc["top1"],
    "Corredor con mayor carga por tiempo (red arterial, k=1000)": top_k,
    "Corredor con mayor carga por tiempo (intermediación exacta)": corr["carga_t"].idxmax(),
    "Corredor con mayor carga por distancia": corr["carga_l"].idxmax(),
    "Corredor con mayor carga (red completa)": corr_drv["carga"].idxmax(),
    "Top 10 corredores por carga por tiempo (intermediación exacta)": "; ".join(top10),
    "Spearman entre pesos tiempo y distancia (tramos)": rho_pesos,
    "Jaccard top 10 corredores entre pesos": jac,
    "Tiempo intermediación red completa k=1000 (min)": min_drv,
    "Dependencia de velocidad NQS": dv["Avenida Carrera 30"],
    "Dependencia de velocidad Transversal 35": dv["Transversal 35"],
    "Dependencia de velocidad Av. Ciudad de Cali (Av. Carrera 86)": dv["Avenida Carrera 86"],
    "Dependencia de velocidad Av. Boyacá (Av. Carrera 72)": dv["Avenida Carrera 72"],
    "Dependencia de velocidad Av. Carrera 68": dv["Avenida Carrera 68"],
    "Dependencia de velocidad Av. Carrera 50": dv["Avenida Carrera 50"],
    "Tipo de corredor de la NQS": dep.loc["Avenida Carrera 30", "tipo_avance"],
    "Spearman de carga entre red arterial y completa": comp["spearman"],
    "Eficiencia global base E0": E0,
    "Caída de eficiencia al cerrar la NQS (%)": cierres.set_index("corredor").loc["Avenida Carrera 30", "caida_E_%"],
    "Caída de eficiencia con 5% de tramos retirados, ataque adaptativo (%)": caida_5,
    "Fracción de colapso, ataque adaptativo": f"{fc['adaptativo']:.1%}",
    "Fracción de colapso, fallas al azar": f"{fc['azar']:.1%}",
    "Macrozonas tras 7 pasos de ataque adaptativo (nodos)": ", ".join(map(str, macrozonas)),
    "Fracción en la componente débil mayor tras el ataque": f"{debil:.1%}",
    "Robustez base R0 (una réplica)": R0,
    "Eficiencia con intervención A, enlace 4,2 km (%)": res_esc.set_index("escenario").loc[nombre_A, "dE_%"],
    "Velocidad media Av. Carrera 68 (km/h)": vel_actual["Avenida Carrera 68"],
    "Velocidad media Av. Carrera 50 (km/h)": vel_actual["Avenida Carrera 50"],
    "Tiempo entre zonas con A (%)": tiempo[nombre_A],
    "Tiempo entre zonas con B (%)": tiempo["B Cra 68 y Cra 50 a 50 km/h"],
    "Carga de la NQS con B (%)": carga_nqs["B"],
    "Carga de la NQS con B por km intervenido (%/km)": por_km["B"],
    "Carga de la NQS con A (%)": carga_nqs["A"],
    "Cambio de R con A, media de 3 réplicas (%)": res_final.set_index("escenario").loc["A", "dR_%"],
    "Cambio de R con B, media de 3 réplicas (%)": res_final.set_index("escenario").loc["B", "dR_%"],
    "Desviación estándar de R entre réplicas": (f"{res_final['R_std'].min():.3f} a "
                                                f"{res_final['R_std'].max():.3f}"),
    "Eficiencia con intervención B (%)": res_final.set_index("escenario").loc["B", "dE_%"],
    "Cambio de R con A3+B (%)": res_final.set_index("escenario").loc["A+B", "dR_%"],
    "Eficiencia con A3+B (%)": res_final.set_index("escenario").loc["A+B", "dE_%"],
}
NO_SE_COMPARAN = {"Tiempo intermediación red completa k=1000 (min)"}   # costo, no resultado


def decimales(texto: str) -> int:
    return len(texto.split(".")[1]) if "." in texto else 0


def coincide(valor, base: str) -> bool:
    try:
        return round(float(valor), decimales(base)) == float(base)
    except (TypeError, ValueError):
        return str(valor) == base


ref = pd.read_csv(RUTAS["referencia"], dtype=str, keep_default_na=False)
faltan = set(calculado) - set(ref["metrica"])
if faltan:
    raise KeyError(f"Métricas sin fila en la línea base: {faltan}")

filas = []
for metrica, valor in calculado.items():
    base = ref.loc[ref["metrica"] == metrica, "valor_linea_base"].iloc[0]
    texto = f"{valor:.6g}" if isinstance(valor, float) else str(valor)
    ref.loc[ref["metrica"] == metrica, "valor_refactor"] = texto
    if metrica in NO_SE_COMPARAN:
        estado = "no se compara"
    elif base in ("sin verificar", "no calculado"):
        estado = "nuevo: la línea base no lo verificó"
    else:
        estado = "coincide" if coincide(valor, base) else "DIFIERE"
    filas.append({"metrica": metrica, "linea_base": base, "refactor": texto, "estado": estado})

verificacion = pd.DataFrame(filas)
ref.to_csv(RUTAS["referencia"], index=False)
print(verificacion["estado"].value_counts().to_string())
verificacion

### Archivos completos frente a las salidas del avance

Las curvas, los cierres y el registro del ataque se comparan fila por fila con los CSV que guardó el notebook del avance.

In [ ]:
def comparar_tabla(nuevo: pd.DataFrame, archivo: str, claves: list[str], valores: list[str]) -> dict:
    base = pd.read_csv(RUTAS["grafos"] / archivo)
    for t in (base, nuevo):
        if "f" in claves:
            t["f"] = t["f"].round(6)
    m = base.merge(nuevo, on=claves, how="outer", suffixes=("_base", "_nuevo"), indicator=True)
    sin_par = int((m["_merge"] != "both").sum())
    dif = max((m[f"{v}_base"] - m[f"{v}_nuevo"]).abs().max() for v in valores)
    return {"archivo": archivo, "filas_base": len(base), "filas_sin_par": sin_par,
            "max_diferencia": dif, "estado": "coincide" if sin_par == 0 and dif < 1e-9 else "DIFIERE"}


base_log = pd.read_csv(RUTAS["grafos"] / "log_ataque_recalculado.csv")
nuevo_log = registro_ad[registro_ad["paso"] <= base_log["paso"].max()]
pares_base, pares_nuevo = set(map(tuple, base_log.values)), set(map(tuple, nuevo_log.values))

archivos = pd.DataFrame([
    comparar_tabla(curvas[["f", "S", "E_rel", "estrategia", "rep"]].copy(), "curvas_resiliencia.csv",
                   ["estrategia", "rep", "f"], ["S", "E_rel"]),
    comparar_tabla(cierres.copy(), "cierre_corredores.csv", ["corredor"],
                   ["km", "caida_E_%", "control_aleatorio_%", "veces_control"]),
    {"archivo": "log_ataque_recalculado.csv", "filas_base": len(base_log),
     "filas_sin_par": len(pares_base ^ pares_nuevo), "max_diferencia": 0.0,
     "estado": "coincide" if pares_base == pares_nuevo else "DIFIERE"},
])
archivos

In [ ]:
nombres_avance = {"A": "A3", "A+B": "A3+B"}
archivos_int = pd.DataFrame([
    comparar_tabla(res_esc.copy(), "escenarios_robustez.csv", ["escenario"], ["R", "dR_%", "dE_%"]),
    comparar_tabla(res_final.replace({"escenario": nombres_avance}).drop(columns="dR_pareado_%"),
                   "escenarios_final.csv", ["escenario"], ["R_media", "R_std", "dE_%", "dR_%"]),
])
archivos_int

## 13. Definiciones del informe (A5, A6, A9 y A13)

Hasta aquí el notebook reproduce el avance. Esta sección aplica las definiciones que cambian cifras:

- **A6:** cortes del índice de dependencia de velocidad en 0,6 y 2.
- **A13:** un tramo con nombre compuesto cuenta para cada corredor que nombra.
- **A5 y A9:** f_c donde S cae a 0,5, con pasos de 0,5%, en fracción nominal y efectiva.

Las cifras de esta sección son las que usa el informe. Cada diferencia frente a la versión del avance queda en `cambios_definiciones.csv`.

In [ ]:
corr_inf = pr.corredores(tramos, {"carga_t": "eb", "carga_l": "eb_len"},
                         min_km=PARAM["min_km_corredor"], dividir_compuestos=True)
dep_inf = pr.dependencia_velocidad(corr_inf, "carga_t", "carga_l",
                                   PARAM["cortes_a6"], PARAM["etiquetas_a6"])
corr_drv_inf = pr.corredores(tramos_drv, {"carga": "eb"},
                             min_km=PARAM["min_km_corredor"], dividir_compuestos=True)
comp_inf = pr.comparar_corredores(dep_inf["carga_t"], corr_drv_inf["carga"], n=PARAM["top_n"])
top10_inf = list(dep_inf["carga_t"].nlargest(PARAM["top_n"]).index)
dep_inf.to_csv(RUTAS["salida"] / "corredores_arterial_informe.csv")

cambios = [
    {"cifra": "Corredores de 2 km o más", "avance": len(dep), "informe": len(dep_inf)},
    {"cifra": "Top 10 por carga por tiempo", "avance": "; ".join(top10), "informe": "; ".join(top10_inf)},
    {"cifra": "Spearman de carga entre redes", "avance": round(comp["spearman"], 3),
     "informe": round(comp_inf["spearman"], 3)},
    {"cifra": "Corredores comunes entre redes", "avance": comp["comunes"], "informe": comp_inf["comunes"]},
    {"cifra": "Jaccard top 10 entre pesos", "avance": round(jac, 3),
     "informe": round(pr.jaccard_top(dep_inf["carga_t"], dep_inf["carga_l"], n=PARAM["top_n"]), 3)},
]
for etiqueta, nombre in CORREDORES_CITADOS.items():
    cambios.append({"cifra": f"Dependencia de velocidad {etiqueta}",
                    "avance": f"{dep.loc[nombre, 'dep_vel']:.3f} ({dep.loc[nombre, 'tipo_avance']})",
                    "informe": f"{dep_inf.loc[nombre, 'dep_vel']:.3f} ({dep_inf.loc[nombre, 'tipo']})"})
cambios = pd.DataFrame(cambios)
cambios["cambia"] = cambios["avance"].astype(str) != cambios["informe"].astype(str)
cambios.to_csv(RUTAS["salida"] / "cambios_definiciones.csv", index=False)

print(f"Tramos sin nombre excluidos: {excluidos['tramos']} ({excluidos['km']:.1f} km)")
print("Tipos de corredor (A6):", dep_inf["tipo"].value_counts().to_dict())
cambios

In [ ]:
AI = PARAM["ataque_informe"]
curvas_inf = [pr.ataque(G_art, "aleatorio", fuentes_ef, E0, AI["pasos"], AI["frac"], seed=s)[0]
              for s in range(A["reps_azar"])]
curvas_inf.append(pr.ataque(G_art, "eb_recalculada", fuentes_ef, E0, AI["pasos"], AI["frac"],
                            k_recalculo=A["k_recalculo"])[0])
curvas_inf = pd.concat(curvas_inf, ignore_index=True)
curvas_inf.to_csv(RUTAS["salida"] / "curvas_resiliencia_informe.csv", index=False)

ad_inf = curvas_inf[curvas_inf["estrategia"] == "eb_recalculada"]
azar_inf = curvas_inf[curvas_inf["estrategia"] == "aleatorio"].groupby("f")[["S", "f_efectiva"]].mean()
fc_tabla = pd.DataFrame({
    "presentado": {"adaptativo": "6,5%", "azar": "14%"},
    "nominal, pasos de 1%": {k: f"{v:.1%}" for k, v in fc.items()},
    "efectiva, pasos de 1%": {k: f"{v:.1%}" for k, v in fc_ef.items()},
    "nominal, pasos de 0,5% (A5)": {
        "adaptativo": f"{pr.f_critica(ad_inf['f'], ad_inf['S'], u):.1%}",
        "azar": f"{pr.f_critica(azar_inf.index, azar_inf['S'], u):.1%}"},
    "efectiva, pasos de 0,5% (A5, A9)": {
        "adaptativo": f"{pr.f_critica(ad_inf['f_efectiva'], ad_inf['S'], u):.1%}",
        "azar": f"{pr.f_critica(azar_inf['f_efectiva'], azar_inf['S'], u):.1%}"},
})
fc_tabla.to_csv(RUTAS["salida"] / "fc_informe.csv")
fc_tabla